## Final Section, from here we will output the `trucks required` for each `hour` of the predictions and for each `plant`.

### Previous Step: Modeling (first or second iteration)

### Next Step: Finish Documenting Paper

---
#### TODO:

- `Analyze truck patterns and behaviour through time and plants`
- `Predict the used trucks per plant for the 45 days horizon`
- `Based on predicted remissions and volume, as well as analyzed truck delivery patterns and requested ideal cargo per truck, calculate and graph per plant the amount of trucks required per hour and per plant` 

- There have been `149` trucks:
    + TODO: Verify unused trucks
    + TODO: Verify truck's routes and activity
    + TODO: Verify how new trucks or trucks ceasing opreations affect the data

In [ ]:
# Load cleaned data

import plotly
import plotly.express as px

In [ ]:
# Check for outliers (first trucks)


#See ship_plant_code frequency, which is a categorical variable

fig_plant_frequency = plotly.express.histogram(remissions_cleaned, x=remissions_cleaned['ship_plant_code'].astype(str), title='Distribution of ship_plant_code')
fig_plant_frequency.show()


In [ ]:
# Categorical data distribution: Truck codes distribution
truck_codes_distribution = (
    remissions_EDA['truck_code']
    .value_counts()
    .rename_axis('truck_code')
    .reset_index(name='count')
)

truck_codes_distribution['percentage'] = (
    truck_codes_distribution['count'] / len(remissions_EDA) * 100
).round(4)

truck_codes_distribution.head(-1)

In [ ]:
fig_truck_frequency = plotly.express.histogram(remissions_cleaned, x=remissions_cleaned['truck_code'].astype(str), title="Frequency of orders per truck")
fig_truck_frequency.update_layout(xaxis={'categoryorder':'total descending'})
fig_truck_frequency.show()

In [ ]:
# Date since each truck started working with GCC
remissions_cleaned['order_date'] = pd.to_datetime(remissions_cleaned['order_date'], errors='raise')
remissions_cleaned['truck_code'] = remissions_cleaned['truck_code'].astype(str)
truck_start_dates = remissions_cleaned.groupby('truck_code')['order_date'].min().reset_index()
truck_start_dates = truck_start_dates.sort_values(by='order_date', ascending=True)

fig_truck_first_dates = plotly.express.histogram(truck_start_dates, x=truck_start_dates['order_date'], title="Distribution of first order dates for each truck", nbins=40)
fig_truck_first_dates.show()

#Most trucks where first seen since the beginning of the data, a few have been purchased each year that passes.


In [ ]:
# Order trucks by months since last order and plot each truck individually (sorted)
truck_last_dates = remissions_cleaned.groupby('truck_code')['order_date'].max().reset_index()
truck_last_dates.columns = ['truck_code', 'last_order_date']
max_date_in_db = remissions_cleaned['order_date'].max()
truck_last_dates['months_since_last_order'] = ((max_date_in_db - truck_last_dates['last_order_date']).dt.days / 30).round(1)
truck_last_dates = truck_last_dates.sort_values('months_since_last_order', ascending=False).reset_index(drop=True)

# Full ordered bar chart (may be crowded for many trucks)
fig_truck_months = plotly.express.bar(
    truck_last_dates,
    x='truck_code',
    y='months_since_last_order',
    title='Months since last order per truck (ordered)',
    text='months_since_last_order'
)
fig_truck_months.update_traces(texttemplate='%{text}', textposition='outside', marker_color='indianred')
fig_truck_months.update_layout(xaxis_tickangle=90, xaxis={'categoryorder':'array','categoryarray':truck_last_dates['truck_code'].tolist()})
fig_truck_months.show()

# Alternative: show top N most inactive trucks for readability
top_n = 30
fig_top = plotly.express.bar(
    truck_last_dates.head(top_n),
    x='truck_code',
    y='months_since_last_order',
    title=f'Top {top_n} trucks by months since last order',
    text='months_since_last_order'
)
fig_top.update_traces(texttemplate='%{text}', textposition='outside', marker_color='darkorange')
fig_top.update_layout(xaxis_tickangle=90)
fig_top.show()

In [ ]:
#Show a relation of the truck's time of first appearance and the amount of orders they have made.
truck_order_counts = remissions_cleaned['truck_code'].value_counts().reset_index()
truck_order_counts.columns = ['truck_code', 'order_count'] 
truck_order_counts = truck_order_counts.merge(truck_start_dates, on='truck_code', how='left')
truck_order_counts = truck_order_counts.sort_values(by='order_count', ascending=False)
truck_order_counts = truck_order_counts.rename(columns={'order_date': 'first_order_date'})
fig_truck_orders_vs_first_date = plotly.express.scatter(
    truck_order_counts,
    x='first_order_date',
    y='order_count',
    title="Relation between truck's first order date and total orders"
)
fig_truck_orders_vs_first_date.show()

In [ ]:
# Merge truck activity metrics for heatmap
truck_activity = truck_order_counts.merge(
    truck_last_dates[['truck_code', 'months_since_last_order']],
    on='truck_code',
    how='left'
)

# Enforce a minimum visible size
truck_activity['bubble_size'] = truck_activity['order_count'].clip(lower=8)

# Merge truck activity metrics for heatmap
truck_activity = truck_order_counts.merge(
    truck_last_dates[['truck_code', 'last_order_date', 'months_since_last_order']],
    on='truck_code',
    how='left'
)

# Enforce a minimum visible size
truck_activity['bubble_size'] = truck_activity['order_count'].clip(lower=8)

fig_heatmap = plotly.express.scatter(
    truck_activity,
    x='months_since_last_order',
    y='order_count',
    color='order_count',
    size='bubble_size',
    hover_data=['truck_code', 'first_order_date', 'last_order_date'],
    title='Heat map: Truck Activity (Orders vs Months Since Last Order)',
    labels={
        'order_count': 'Total Orders',
        'months_since_last_order': 'Months Since Last Order'
    },
    color_continuous_scale='Viridis'
)
fig_heatmap.update_layout(height=600, width=900)
fig_heatmap.show()


In [ ]:
# Remission counts and percentages per truck (tail)
total_remissions = len(remissions_cleaned)

truck_remissions = (
    remissions_cleaned.groupby('truck_code')['tkt_code']
    .agg(list)
    .reset_index(name='tkt_codes')
)

truck_remissions['remission_count'] = truck_remissions['tkt_codes'].str.len()
truck_remissions['completion_percentage'] = (
    truck_remissions['remission_count'] / total_remissions * 100
).round(4)

truck_remissions = truck_remissions.sort_values(
    'completion_percentage', ascending=False
).reset_index(drop=True)

truck_remissions.tail(20)